# Lesson 5: Stock Market Analysis with LLM Feedback Loops

## Understanding LLM-to-LLM Feedback Systems

This notebook demonstrates the core concept of LLM feedback loops where:
1. **Analyst LLM** generates initial stock analysis
2. **Critic LLM** evaluates and provides specific feedback
3. **Analyst LLM** refines its analysis based on critic feedback
4. Process repeats until quality criteria are met

**Time Required**: 15-20 minutes
**Context**: Stock market analysis and investment recommendations

## Setup and Configuration

In [1]:
# Setup - Import libraries and configure environment
import os
from dotenv import load_dotenv
from openai import OpenAI
import re

# Load environment variables
load_dotenv('../../.env')

# Setup OpenAI client
client = OpenAI(
    base_url="https://openai.vocareum.com/v1",
    api_key=os.getenv("OPENAI_API_KEY")
)

print("✅ Environment configured successfully")
print("🎯 Ready to demonstrate LLM feedback loops in investment advisory")

print("✅ Environment setup complete")

✅ Environment configured successfully
🎯 Ready to demonstrate LLM feedback loops in investment advisory
✅ Environment setup complete


## Demo Scenario: Apple Stock Analysis

We'll analyze Apple (AAPL) stock using a two-LLM feedback system:
- **Stock Analyst LLM**: Generates investment analysis and recommendations
- **Investment Critic LLM**: Reviews and provides improvement feedback

In [2]:
# Sample market data for Apple (AAPL) - simplified for demo
market_data = """
Company: Apple Inc. (AAPL)
Current Price: $175.50
52-Week Range: $124.17 - $199.62
Market Cap: $2.75T
P/E Ratio: 28.5
Recent News: 
- iPhone 15 launch exceeded expectations
- Services revenue grew 16% YoY
- China sales down 4% due to local competition
- New Vision Pro headset launching next year
- Strong cash position of $162B
"""

print("📊 Market data loaded for Apple (AAPL)")
print(market_data)

📊 Market data loaded for Apple (AAPL)

Company: Apple Inc. (AAPL)
Current Price: $175.50
52-Week Range: $124.17 - $199.62
Market Cap: $2.75T
P/E Ratio: 28.5
Recent News: 
- iPhone 15 launch exceeded expectations
- Services revenue grew 16% YoY
- China sales down 4% due to local competition
- New Vision Pro headset launching next year
- Strong cash position of $162B



## Step 1: Define the LLM Agents

In [ ]:
# TODO: Define the Stock Analyst LLM prompt
# This LLM should act as a Senior Stock Analyst
# It should analyze stocks and provide investment recommendations
# Include instructions for: key strengths/risks, price target, recommendation
ANALYST_PROMPT = """
Act as a Senior Stock Analyst. Analyse stocks and provide investment recommendations.
Instructions:  
- Provide a clear summary of the stock's key strengths and risks.
- Use only provided data
- Length should be 2-3 paragraphs
- Suggest a price target based on the analysis.
- Provide a recommendation: Buy, Hold, or Sell.
"""

# TODO: Define the Investment Critic LLM prompt  
# This LLM should act as a Senior Investment Critic and Risk Manager
# It should review analyses and provide constructive feedback
# Include criteria for approval (quality score 8+) and specific feedback areas
CRITIC_PROMPT = """
Act as a Senior Investment Critic and Risk Manager. Review investment analyses and provide constructive feedback.
Instructions:
- Evaluate the quality of the analysis (score out of 10).
- Use only provided data
- Provide specific feedback on strengths and weaknesses.
- Suggest improvements or additional considerations.

Output-  The first two lines of response should read 
- VERDICT : APPROVED / REVISION REQUIRED (only approved if 8 or higher out of 10)
- SCORE : x/10
"""

print("🤖 LLM Agents defined:")
print("- Stock Analyst: Generates investment analysis")
print("- Investment Critic: Reviews and provides feedback")

🤖 LLM Agents defined:
- Stock Analyst: Generates investment analysis
- Investment Critic: Reviews and provides feedback


## Step 2: Helper Functions for LLM Interaction

In [4]:
def get_analyst_analysis(market_data, previous_feedback=None, previous_analysis=None):
    """Get stock analysis from the Analyst LLM"""
    
    # TODO: Create user prompt with market data
    user_prompt = f"Market Data:\n{market_data}"
    
    # TODO: If there's previous feedback, add it to the prompt
    if previous_feedback:
        if previous_analysis:
            user_prompt += f"\n\nPrevious Analysis:\n{previous_analysis}"
        user_prompt += f"\n\nPrevious Feedback:\n{previous_feedback}"
        user_prompt += "\n\nPlease revise your investment recommendation to address the concerns above in light of the previous analysis."

    # Hint: Check if previous_feedback exists and append instructions to revise
    
    # TODO: Make API call to get analysis
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": ANALYST_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7,
        max_tokens=500
    )
    return response.choices[0].message.content

def get_critic_feedback(analysis, market_data):
    """Get feedback from the Critic LLM"""
    
    # TODO: Create user prompt with both market data and analysis to review
    user_prompt = f"""
    Market Data:\n{market_data}
    Analysis:\n{analysis}
    """
    
    # TODO: Make API call to get feedback
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": CRITIC_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.3,
        max_tokens=800
    )
    return response.choices[0].message.content

def is_approved(feedback: str) -> bool:
    text = feedback.upper()
    return "APPROVED" in text and "NOT APPROVED" not in text


def extract_score(feedback: str):
    """Return the critic's score as an int (0-10), or None if it can't be found."""
    match = re.search(r"SCORE\s*\**\s*:\s*\**\s*(\d{1,2})(?:\.\d+)?\s*/\s*10", feedback, re.IGNORECASE)
    return int(match.group(1)) if match else None


print("🔧 Helper functions created for LLM interactions")

🔧 Helper functions created for LLM interactions


## Step 3: Demonstration - Single Feedback Loop

In [5]:
print("🚀 Starting LLM Feedback Loop Demo...\n")
print("="*60)
print("ROUND 1: Initial Analysis")
print("="*60)

# TODO: Get initial analysis from Analyst LLM
initial_analysis = get_analyst_analysis(market_data)

print("📈 ANALYST OUTPUT:")
print("-" * 40)
print(initial_analysis)
print()

🚀 Starting LLM Feedback Loop Demo...

ROUND 1: Initial Analysis
📈 ANALYST OUTPUT:
----------------------------------------
Apple Inc. (AAPL) continues to demonstrate strong operational strengths, particularly in its flagship product, the iPhone. The recent launch of the iPhone 15 has exceeded market expectations, which bodes well for future revenue growth and customer loyalty. Additionally, Apple's Services segment, which includes revenue from the App Store, Apple Music, and iCloud, grew by an impressive 16% year-over-year, indicating a successful shift towards recurring revenue streams that are less dependent on hardware sales. Furthermore, Apple's robust cash position of $162 billion provides a solid foundation for future investments and potential shareholder returns through buybacks or dividends.

However, there are notable risks that investors should consider. A 4% decline in sales from China due to increased competition raises concerns about Apple's ability to maintain its market 

In [6]:
# TODO: Get feedback from Critic LLM
print("🔍 CRITIC REVIEW:")
print("-" * 40)

critic_feedback = get_critic_feedback(initial_analysis, market_data)  # Replace with function call
print(critic_feedback)
print()

# TODO: Check if the analysis was approved
# Hint: Look for "APPROVED" in the feedback text
approved = is_approved(critic_feedback)
print(f"✅ Status: {'APPROVED' if approved else 'NEEDS REVISION'}")

🔍 CRITIC REVIEW:
----------------------------------------
VERDICT : REVISION REQUIRED  
SCORE : 7/10  

Strengths:  
- The analysis effectively highlights Apple's operational strengths, particularly the successful launch of the iPhone 15 and the growth of the Services segment, which is crucial for long-term sustainability.  
- The mention of Apple's strong cash position is a significant positive, as it provides flexibility for investments and shareholder returns.  
- The recommendation to hold is well-supported by the analysis of both strengths and risks, demonstrating a balanced view.

Weaknesses:  
- The analysis could benefit from a more detailed examination of the competitive landscape in China, including specific competitors and their market strategies.  
- The potential impact of the Vision Pro headset launch could be elaborated further, including market expectations and consumer interest metrics.  
- The analysis does not address macroeconomic factors that could influence Apple’

## Step 4: Feedback Loop - Iterative Improvement

In [7]:
# TODO: Implement the revision logic
# If not approved, get revised analysis and final review
if not approved:
    print("\n" + "="*60)
    print("ROUND 2: Revised Analysis Based on Feedback")
    print("="*60)
    
    # TODO: Get revised analysis incorporating feedback
    revised_analysis = get_analyst_analysis(market_data, critic_feedback)  # Replace with function call
    
    print("📈 REVISED ANALYST OUTPUT:")
    print("-" * 40)
    print(revised_analysis)
    print()
    
    # TODO: Get final review
    print("🔍 FINAL CRITIC REVIEW:")
    print("-" * 40)
    
    final_feedback = get_critic_feedback(revised_analysis, market_data)  # Replace with function call
    print(final_feedback)
    print()
    
    # TODO: Check final approval status
    final_approval = is_approved(final_feedback)
    print(f"✅ Final Status: {'APPROVED' if final_approval else 'NEEDS FURTHER REVISION'}")
else:
    print("\n🎉 Analysis approved on first iteration!")


ROUND 2: Revised Analysis Based on Feedback
📈 REVISED ANALYST OUTPUT:
----------------------------------------
**Stock Analysis for Apple Inc. (AAPL)**

Apple Inc. (AAPL) continues to showcase its operational prowess, highlighted by the successful launch of the iPhone 15, which has exceeded market expectations. The company's services revenue, growing at 16% year-over-year, demonstrates a strong shift towards high-margin recurring revenue streams, further solidifying its long-term growth potential. Additionally, Apple's robust cash position of $162 billion provides it with substantial financial flexibility to invest in innovation, acquisitions, and shareholder returns through dividends and buybacks. However, the company faces notable challenges, particularly in the competitive landscape in China, where recent sales dipped by 4% due to local competition, signaling potential vulnerabilities in one of its largest markets. The anticipated launch of the Vision Pro headset next year presents

## Step 5: Complete Feedback Loop Function

In [8]:
def stock_analysis_with_feedback(market_data,max_iterations=5):
    """
    Complete LLM feedback loop for stock analysis
    
    Returns:
        dict: Final analysis results and iteration history
    """
    
    # TODO: Initialize results dictionary
    results = {
        'iterations': [],
        'final_analysis': None,
        'approved': False,
        'total_iterations': 0
    }
    
    analysis = None
    feedback = None
    
    # TODO: Implement the feedback loop
    for iteration in range(max_iterations):
        print(f"\n🔄 Iteration {iteration + 1}")
        print("-" * 30)
        
        # TODO: Get analysis (initial or revised)
        if iteration == 0:
            analysis = get_analyst_analysis(market_data)
        else:
            analysis = get_analyst_analysis(market_data, feedback, analysis)
        # TODO: Get critic feedback
        feedback = get_critic_feedback(analysis, market_data)
        # TODO: Store iteration results
        results['iterations'].append({
            'iteration': iteration + 1,
            'analysis': analysis,
            'feedback': feedback
        })
        score = extract_score(feedback)
        score_text = f"{score}/10" if score is not None else "score n/a"
        results['total_iterations'] += 1

        
        # TODO: Check if approved and break if so
        if is_approved(feedback):
            results['approved'] = True
            print(f"✅ Analysis approved ({score_text}) on iteration {results['total_iterations']}")
            break
        
        # TODO: Continue if not approved
        if not is_approved(feedback) and iteration < max_iterations - 1:
            print(f"❌ Analysis not approved ({score_text}), revising...(iteration {results['total_iterations']})")
        else:
            print(f"❌ Analysis not approved, reached max iterations. Final analysis may not be approved.")
    
    # TODO: Set final results
    results['final_analysis'] = analysis

    return results

print("🔧 Complete feedback loop function ready")

🔧 Complete feedback loop function ready


## Step 6: Test Different Scenarios

In [ ]:
# Test with a more challenging scenario
challenging_data = """
Company: Tesla Inc. (TSLA)
Current Price: $248.50
52-Week Range: $138.80 - $299.29
Market Cap: $790B
P/E Ratio: 65.2
Recent News:
- Q3 deliveries missed expectations by 2%
- Cybertruck production delays continue
- FSD (Full Self Driving) beta expanding
- Price cuts in China market
- Musk selling more shares for Twitter
- Competition increasing from Ford, GM
"""

print("🧪 Testing with more challenging scenario: Tesla (TSLA)")
print("This scenario has mixed signals and should trigger more feedback loops...\n")

# TODO: Test the complete feedback loop function
tesla_results = stock_analysis_with_feedback(challenging_data, max_iterations=5)

# TODO: Print summary results
print(f"\n📊 Summary:")
print(f"- Total iterations: {tesla_results['total_iterations']}")
print(f"- Final approval: {tesla_results['approved']}")
print(f"- Improvement demonstrated: {'Yes' if tesla_results['total_iterations'] > 1 else 'No'}")

🧪 Testing with more challenging scenario: Tesla (TSLA)
This scenario has mixed signals and should trigger more feedback loops...


🔄 Iteration 1
------------------------------
❌ Analysis not approved (7/10), revising...(iteration 1)

🔄 Iteration 2
------------------------------
❌ Analysis not approved (7/10), revising...(iteration 2)

🔄 Iteration 3
------------------------------
❌ Analysis not approved (7/10), revising...(iteration 3)

🔄 Iteration 4
------------------------------
❌ Analysis not approved (7/10), revising...(iteration 4)

🔄 Iteration 5
------------------------------


## Key Concepts Demonstrated

### 1. LLM-to-LLM Feedback Loop
- **Analyst LLM** generates initial analysis
- **Critic LLM** evaluates and provides specific feedback
- **Analyst LLM** incorporates feedback for improvement
- Process continues until quality criteria are met

### 2. Real-World Applications
- **Investment Research**: Multi-step analysis refinement
- **Risk Assessment**: Peer review and validation
- **Report Generation**: Quality control and improvement
- **Decision Support**: Collaborative AI reasoning

### 3. Benefits of This Approach
- **Quality Improvement**: Iterative refinement leads to better outputs
- **Error Reduction**: Critic LLM catches missing factors or errors
- **Consistency**: Systematic evaluation criteria
- **Transparency**: Clear improvement trail and reasoning

## Exercise: Try Your Own Stock

Now try the feedback loop with a different stock scenario:

In [ ]:
# TODO: Create your own stock scenario and test the feedback loop
your_stock_data = """
Company: [Your Company Name]
Current Price: $X.XX
52-Week Range: $X.XX - $X.XX
Market Cap: $XB
P/E Ratio: X.X
Recent News:
- [Add 3-5 relevant news items]
- [Mix positive and negative factors]
- [Include some ambiguous information]
"""

# TODO: Test with your stock data
# your_results = stock_analysis_with_feedback(your_stock_data)
# print(f"Your stock analysis completed in {your_results['total_iterations']} iterations")

print("💡 Create your own stock scenario above and test the feedback system!")

## Summary

This exercise illustrated the core concept of **LLM feedback loops** in financial analysis:

✅ **LLM-to-LLM Communication**: One LLM evaluates another's output  
✅ **Iterative Improvement**: Analysis gets refined through feedback cycles  
✅ **Quality Control**: Systematic evaluation criteria and approval process  
✅ **Real-World Application**: Stock analysis with practical financial context  

**Key Takeaway**: Unlike traditional ML training, this approach uses LLMs to provide contextual, reasoning-based feedback that leads to immediate output improvement - demonstrating true AI-to-AI collaboration in financial decision-making.

---
*Time: ~15-20 minutes | Context: Stock Market Analysis | Concept: LLM Feedback Loops*